In [5]:
import time
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

def fetch_banapresso():
    url = "https://www.banapresso.com/"
    
    driver = webdriver.Chrome()
    driver.maximize_window()
    driver.get(url)
    wait = WebDriverWait(driver, 10)
    time.sleep(1)

    # 1. 매장 검색 화면까지 진입
    action = ActionChains(driver)
    
    first_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#wrap > header > div > ul > li:nth-child(2) > a > span"
    )

    second_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#wrap > header > div > ul > li:nth-child(2) > ul > li:nth-child(1) > a > span"
    )

    # 마우스 이동
    action.move_to_element(first_tag).move_to_element(second_tag).click().perform()
    # 1 ==========================

    # 2. 위치 권한 설정 확인 없애기
    store_shop_list = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#store > div > div.sc-24af01fc-0.sc-24af01fc-2.kRHEaB.EbNMr.store_list_box > div.sc-24af01fc-6.drblfU.store_shop_list"))
    )
    time.sleep(1)

    location_permissions = driver.find_element(By.CSS_SELECTOR, "#root > div.sc-ff716c63-0.fOAigp > div > div.p_btm > button")

    if location_permissions:
        location_permissions.click()
    else:
        pass
    time.sleep(1)
    # 2 ============================

    # 3. 리스트 스크롤로 끝까지 내리기
    scroll_ul = driver.find_element(By.CSS_SELECTOR, "#store > div > div.sc-24af01fc-0.sc-24af01fc-2.kRHEaB.EbNMr.store_list_box > div.sc-24af01fc-6.drblfU.store_shop_list > ul")
    list_bottom = driver.find_element(By.CSS_SELECTOR, "#store > div > div.sc-24af01fc-0.sc-24af01fc-2.kRHEaB.EbNMr.store_list_box > div.sc-24af01fc-6.drblfU.store_shop_list > ul > li.sc-24af01fc-10.laawiM.observer")

    elapsed_time = 0
    while True:
        before = time.time()
        action.move_to_element(scroll_ul).scroll_to_element(list_bottom).perform()
        after = time.time()
        elapsed_time += after - before
        if elapsed_time >= 8:
            break
    # 3 ===============================

    # 4. 웹 크롤링
    html = driver.page_source

    soup = BeautifulSoup(html, "html.parser")

    stores = soup.select('div.store_shop_list ul li')
    print(len(stores))
    data = []
    for store in stores:
        
        name_tag = store.select_one("p.name")
        if name_tag is None: break
        address_tag = store.select_one("p.address")
        open_tag = store.select_one("div.time.open p:nth-child(1)")
        opentime_tag = store.select_one("div.time.open p:nth-child(2)")
        parking_tag = store.select_one("div.parking span")


        name = name_tag.get_text(strip=True)
        address = address_tag.get_text(strip=True)
        open = open_tag.get_text(strip=True) if open_tag else ""
        opentime = opentime_tag.get_text(strip=True) if opentime_tag else ""
        parking = parking_tag.get_text(strip=True) if parking_tag else ""

        data.append({
            "매장명": name,
            "주소": address,
            "영업중": open,
            "영업시간": opentime,
            "주차": parking
        })

    df = pd.DataFrame(data)
    driver.quit()
    file_name = f"banapresso.csv"
    df.to_csv(file_name, encoding="utf-8-sig")
    
    return df


fetch_banapresso()

232


,매장명,주소,영업중,영업시간,주차
0,가락몰점,"서울특별시 송파구 양재대로 932, 업무동 1층 로비",OPEN,07:30~23:30,
1,가산디지털단지역점,서울시 금천구 가산동 60-3,OPEN,07:00~19:00,
2,가산안양천점,"서울 금천구 가산 디지털2로 127-143, 101호",OPEN,07:00~20:00,
3,가산어반워크점,"서울시 금천구 가산디지털2로 135, 1동 142호",OPEN,07:00~17:30,
4,가산우림라이온스점,서울 금천구 가산디지털1로 168 b131호,OPEN,07:00~17:00,
...,...,...,...,...,...
226,가산한라시그마밸리점,서울특별시 금천구 가산디지털2로 53 1층 106호,,,
227,명동역점,"서울특별시 중구 삼일대로 300, 1층",,,
228,모란역점,경기도 성남시 중원구 광명로 4,,,
229,서여의도점,,,,
